<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/02_1_Segment_Anything_Model_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SAM 2 이미지 객체 분할 - 인터랙티브 웹 UI**

# 0. colab 환경설정

- colab에서 GPU를 사용할 수 있도록 세팅
    - 런타임 > 런타임 유형 변경 > **Python 3** 와 **T4 GPU** 선택

In [1]:
from google.colab import drive
drive.mount('/content/drive')

print("[현재 파일 위치]")
!pwd
print("[현재 디렉토리의 파일 확인]")
!ls

Mounted at /content/drive
[현재 파일 위치]
/content
[현재 디렉토리의 파일 확인]
drive  sample_data


**day 2** 폴더로 이동해주세요.

왼쪽의 **폴더** 아이콘을 클릭하면 경로를 쉽게 볼 수 있습니다.

In [2]:
# 해당 코드는 예시 코드입니다. 본인 환경에 맞게 경로를 수정하여 사용하세요.
%cd /content/drive/MyDrive/day2

/content/drive/.shortcut-targets-by-id/1-vYRZOLYLvUD5Kma-yELIfM0X7TGxy15/day2


## 0-1. 라이브러리 설치와 체크포인트 다운로드
필요한 패키지를 설치하고, SAM 2 체크포인트 파일을 내려받습니다.  
설치가 끝나면 다음 셀에서 라이브러리를 불러올 준비가 됩니다.

In [3]:
# 실습에 필요한 환경 및 라이브러리 설치
!pip install opencv-python matplotlib gradio
!pip install 'git+https://github.com/facebookresearch/sam2.git'

# SAM2 모델 가중치(Checkpoint) 다운로드
!mkdir -p /content/checkpoints/
!wget -P /content/checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-yaoe99sn
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/sam2.git /tmp/pip-req-build-yaoe99sn
  Resolved https://github.com/facebookresearch/sam2.git to commit 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.0 MB/s eta 0:00:00
  Created wheel for SAM-2: filename=sam_2-1.0-cp312-cp312-linux_x86_64.whl size=183669 sha256=f55d40222ce0b412e35b0df2a8df0dc95f16e0625d8c932fc40e1de8b242ca17
  Stored in directory: /tmp/pip-ephem-wheel-cache-n424k42h/wheels/25/a3/8a/abd69dc6a6926b5e75c24810afac36c7b49b5c0f8a100147d6
  Created wheel for iopath: filename=iopath-0.1.10-py3-no

## 0-2. 라이브러리 불러오기와 디바이스 설정
필수 라이브러리를 불러오고, GPU가 있으면 자동으로 사용하도록 설정합니다.  
이 단계는 추론 속도에 직접 영향을 줍니다.

In [4]:
# 2-2. 라이브러리 불러오기와 디바이스 설정
import torch
import numpy as np
import cv2
import gradio as gr
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor


# 연산을 위한 디바이스 선택 (GPU가 있으면 GPU 사용)
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
else:
    device = torch.device("cpu")

print(f"현재 사용 중인 디바이스: {device}")

현재 사용 중인 디바이스: cuda


## 0-3. SAM 2 모델 로드
다운로드한 체크포인트와 설정 파일을 이용해 SAM 2 모델을 메모리에 올립니다.  
완료되면 다음 셀에서 UI와 추론 함수를 정의할 수 있습니다.

In [5]:
# 다운 받은 SAM 2 모델 불러오기
sam2_checkpoint = "/content/checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

print("SAM 2 모델을 메모리에 불러오는 중입니다... 잠시만 기다려주세요.")
sam2_model = build_sam2(model_cfg, sam2_checkpoint, device=device)
predictor = SAM2ImagePredictor(sam2_model)
print("모델 로딩 완료!")

SAM 2 모델을 메모리에 불러오는 중입니다... 잠시만 기다려주세요.
모델 로딩 완료!


# 1. Gradio 인터랙티브 UI
Gradio를 사용하여 웹 UI에서 모델을 사용해보도록 **예측함수**와 **UI**를 정의합니다.  

## 1-1. 예측 함수 정의
`predict_mask` 함수에 SAM 2 추론 로직이 정의됩니다.  

In [ ]:
from utils_sam import mask_overlay
def predict_mask(img, box_str, input_point, input_label):

    # ==================== 예외 처리 로직 ====================
    if img is None:
        return None, None, "먼저 이미지를 업로드해주세요."

    if len(input_point) <= 0:
        input_point = None
        input_label = None

    input_box = None # input_box는 사용자가 좌표로 입력한 바운딩 박스를 의미합니다.

    # 박스 좌표가 입력된 경우, input_box에 [x_min, y_min, x_max, y_max] 형태로 저장합니다.
    if box_str and box_str.strip() != "":
        try:
            coords = [int(c.strip()) for c in box_str.split(",")]
            if len(coords) == 4:
                input_box = np.array(coords)
            else:
                return None, None, "박스 좌표는 반드시 4개의 숫자(x_min, y_min, x_max, y_max)여야 합니다."
        except:
            return None, None, "박스 좌표 형식이 잘못되었습니다. 숫자와 쉼표만 사용하세요. (예시: 100, 100, 300, 300)"

    if input_point is None and input_box is None:
        return img, None, "최소 1개의 점을 클릭하거나 박스 좌표를 입력해주세요."

    if input_box is not None:
        input_box = input_box[None, :]



    # ==================== SAM 2 모델 추론 로직 ====================


    ####### [실습] SAM 2 모델에 원본 이미지를 설정하세요. #######
    predictor.set_image(?)


    ####### [실습] 모델 입력을 넘파이 배열(np.array)로 전환하세요. ######
    input_point = ?(input_point)  # input_point는 사용자가 선택한 점을 의미합니다.
    input_label = ?(input_label)  # input_label은 추가/제거에 대한 라벨정보를 의미합니다.



    ####### [실습] SAM 2 모델에 전달할 인자를 완성하여 추론을 수행하세요. ######
    masks, scores, _ = predictor.predict(
        point_coords=?,
        point_labels=?,
        box=?,
        multimask_output=False, # 여러 개의 마스크를 추론하는 옵션입니다.
    )


    # 원본 이미지에 마스크를 오버레이하여 시각화
    overlay, rgba_img = mask_overlay(img, masks[0], input_point, input_label, input_box)

    return overlay, rgba_img, f"✨ 마스크 생성 완료! (AI 예측 신뢰도: {scores[0]*100:.1f}%)"

## 1-2. Gradio UI 설정하기
`predict_mask` 함수를 `Gradio UI`에 연결하여 레이아웃과 입력/출력 동작을 정의합니다.

In [ ]:
from utils_sam import get_image_info, add_point, clear_prompts, save_output_image


# ==================== 화면 UI 레이아웃 구성 ====================
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("### 🖱️ SAM 2 인터랙티브 객체 추출 실습기")
    gr.Markdown("""
좌측에 이미지를 업로드하고 마우스로 객체를 클릭하면, 우측 화면에 점이 찍힙니다.

이후 '마스크 생성'을 누르세요.
    """)

    input_point = gr.State([])
    input_label = gr.State([])
    extracted_state = gr.State(None)

    with gr.Row(): # Gradio 화면 UI는 Row, Coloumn을 이용해 구성합니다.
        with gr.Column(scale=1): # 첫번째 열 UI 구성
            input_img = gr.Image(label="1. 이미지 업로드 및 클릭 (입력)", interactive=True, type="numpy")
            image_info_ui = gr.Markdown("이미지를 업로드하면 크기 정보가 여기에 표시됩니다.")
            point_type = gr.Radio(choices=["포함할 부분 (Positive)", "제외할 부분 (Negative)"], value="포함할 부분 (Positive)", label="2. 클릭할 점의 성격")
            gr.Markdown("**💡박스 좌표 입력 가이드**: `x_min, y_min, x_max, y_max`")
            box_input = gr.Textbox(label="3. 박스 좌표 (선택사항)", placeholder="예시: 100, 100, 300, 300")

            with gr.Row():
                clear_btn = gr.Button("🗑️ 초기화")
                run_btn = gr.Button("🚀 마스크 생성", variant="primary")

        with gr.Column(scale=1): # 두번째 열 UI 구성
            output_img = gr.Image(label="결과 확인 및 좌표 표시 화면", type="numpy")
            status_text = gr.Textbox(label="시스템 메시지", interactive=False)

            with gr.Row():
                save_btn = gr.Button("💾 누끼 딴 이미지(PNG) 저장 및 다운로드")
            download_file = gr.File(label="배경 투명화 이미지 다운로드", interactive=False)

    # gr.Image 객체와의 상호작용(변경, 선택)시 호출할 함수와 입/출력을 정의합니다.
    input_img.change(get_image_info, inputs=[input_img], outputs=[image_info_ui, input_point, input_label, output_img])
    input_img.select(add_point, inputs=[input_img, point_type, input_point, input_label], outputs=[input_point, input_label, output_img, status_text])

    # gr.Button 객체와 상호작용(클릭)시 호출할 함수와 입/출력을 정의합니다.
    clear_btn.click(clear_prompts, inputs=[input_img], outputs=[input_point, input_label, output_img, status_text])
    run_btn.click(predict_mask, inputs=[input_img, box_input, input_point, input_label], outputs=[output_img, extracted_state, status_text])
    save_btn.click(save_output_image, inputs=[extracted_state], outputs=[download_file, status_text])

## 1-3 Gradio UI 배포하기
이제 Gradio 웹 인터페이스를 배포합니다!  
출력 결과에 표시되는 `Running on public URL` 링크를 눌러 열어주세요.


**사용 방법은 다음과 같습니다.**
1. 이미지를 업로드합니다.
2. 포함할 부분이나 제외할 부분을 클릭하거나, 필요하면 박스 좌표를 입력합니다.
3. `마스크 생성` 버튼으로 결과를 확인하고, 필요하면 `저장` 버튼으로 배경이 제거된 이미지를 다운로드합니다.

In [ ]:
demo.launch(debug=False, share=True)